# 04. 미래 주가 예측 (Recursive Extension)

## 📋 개요
학습된 모델(Tabular 또는 Seq)을 사용하여 미래 주가를 예측합니다.

## ✨ 트랙별 동작

| `active_model` / `active_seq_model` | 로드 경로 | 입력 구성 |
|------|-----------|----------|
| Tabular (`lgbm`, `mlp`, ...) | `data/03_training/` | 단일 행(t) |
| Seq (`gru`) | `data/03_seq/` | seq_len 행 시퀀스 |

두 경로 모두 동일한 `trapezoid_log_close()` 역산 로직과  
동일한 Recursive Extension 철학을 따릅니다.

## 🔧 최근 패치 이력
- **v4.0.0**: GRU(Seq 트랙) 예측 경로 추가. `is_seq_model()` 분기.
- **v3.10.0**: 사다리꼴 모듈화, `pred_log_return` 스키마 분리
- **v3.9.1**: 사다리꼴 적분 보정

## 🔧 Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings

from src.utils.config import (
    load_config, ProjectPaths,
    is_ensemble, is_seq_model, resolve_model_name,
)
from src.utils.trapezoidal import trapezoid_log_close
from src.features.technical import (
    calc_sma, calc_rsi, calc_macd, calc_bollinger, calc_volume_ratio
)

warnings.filterwarnings('ignore')

## 1️⃣ 설정 및 경로

In [ ]:
cfg   = load_config()
paths = ProjectPaths.from_config(cfg)
paths.ensure_dirs()

SAVE_RAW_PREDICTIONS = cfg.get('debug', {}).get('save_raw_predictions', False)

macro_hist_path     = paths.get_macro_parquet()
macro_forecast_path = paths.get_macro_forecast_parquet()
calendar_path       = paths.get_calendar()

# ── 트랙 판별 ─────────────────────────────────────────────────────────
# config.yaml의 active_model이 seq 모델이면 Seq 트랙,
# 아니면 Tabular 트랙을 사용합니다.
# Seq 트랙은 active_seq_model을 별도로 참조합니다.
active_model = cfg.get('active_model', 'lightgbm')
USE_SEQ_TRACK = is_seq_model(active_model)

if USE_SEQ_TRACK:
    # active_model 자체가 gru인 경우
    active_seq = active_model
else:
    # active_model은 Tabular, Seq 트랙 모델은 active_seq_model
    # 04단계에서 Seq 예측을 생성하려면 USE_SEQ_TRACK을 True로 바꾸거나
    # config.yaml의 active_model을 'gru'로 변경하십시오.
    active_seq = cfg.get('active_seq_model', 'gru')

print(f"📌 예측 트랙: {'Seq (' + active_seq + ')' if USE_SEQ_TRACK else 'Tabular (' + active_model + ')'}")
print(f"🔧 debug.save_raw_predictions = {SAVE_RAW_PREDICTIONS}")

## 2️⃣ 매크로 데이터 로드

In [ ]:
df_macro = pd.read_parquet(macro_hist_path)
df_macro['date'] = pd.to_datetime(df_macro['date'])

if macro_forecast_path.exists():
    df_macro_fc = pd.read_parquet(macro_forecast_path)
    df_macro_fc['date'] = pd.to_datetime(df_macro_fc['date'])
    df_macro = pd.concat([df_macro, df_macro_fc], ignore_index=True)
    df_macro = df_macro.drop_duplicates('date')
    print(f"✅ 매크로 통합: {len(df_macro)}일")
else:
    print(f"⚠️  추정값 없이 실측만 사용")

MACRO_COLS = ['kospi', 'usd_krw', 'vix', 'us_return_1d', 'market_regime']
macro_available_cols = [c for c in MACRO_COLS if c in df_macro.columns]
df_macro_lookup = df_macro[['date'] + macro_available_cols].set_index('date')
MACRO_FEATURE_MAP = {col: f'feature_{col}' for col in macro_available_cols}

## 3️⃣ 데이터 로드

In [ ]:
df_features = pd.read_parquet(paths.get_dataset_parquet())
df_features['date'] = pd.to_datetime(df_features['date'])
df_features = df_features.sort_values(['ticker', 'date']).reset_index(drop=True)

last_date = df_features['date'].max()
print(f"   총 행수: {len(df_features):,}  종목: {df_features['ticker'].nunique()}  최신일: {last_date.date()}")

IS_KOSPI_COL = 'feature_is_kospi'
ticker_is_kospi = df_features.groupby('ticker')[IS_KOSPI_COL].last().to_dict() \
    if IS_KOSPI_COL in df_features.columns else {}

df_calendar = pd.read_csv(calendar_path)
df_calendar['date'] = pd.to_datetime(df_calendar['date'])
forecast_end = pd.to_datetime(cfg['calendar']['forecast_end'])
future_dates = df_calendar[
    (df_calendar['date'] > last_date) &
    (df_calendar['date'] <= forecast_end)
]['date'].sort_values().reset_index(drop=True)

if len(future_dates) == 0:
    raise ValueError(
        f"❌ 예측할 미래 영업일이 없습니다.\n"
        f"   최신 데이터: {last_date.date()}\n"
        f"   캘린더 종료일: {df_calendar['date'].max().date()}\n"
        "💡 Tip: 00a_save_trading_days.ipynb에서 기간을 연장하세요."
    )

print(f"   예측 대상: {len(future_dates)}일 ({future_dates.iloc[0].date()} ~ {future_dates.iloc[-1].date()})")

## 4️⃣ 모델 로드

In [ ]:
if USE_SEQ_TRACK:
    # ── Seq 트랙 ──────────────────────────────────────────────────────
    from src.models.gru_model import GRUModel
    seq_model_dir = paths.get_seq_model_dir()
    model = GRUModel.load(str(seq_model_dir))

    target_type      = model.target_type
    seq_len          = model.seq_len
    forecast_horizon = model.forecast_horizon
    feature_cols     = model.feature_list
    CHUNK_SIZE       = forecast_horizon
    NUM_CHUNKS       = int(np.ceil(len(future_dates) / CHUNK_SIZE))

    print(f"✅ GRUModel 로드: {seq_model_dir}")
    print(f"   seq_len={seq_len}, forecast_horizon={forecast_horizon}")

else:
    # ── Tabular 트랙 ──────────────────────────────────────────────────
    model_path   = paths.get_model_path()
    feature_cols = None   # 모델에서 로드

    if is_ensemble(active_model):
        from src.models.ensemble_model import EnsembleModel
        model = EnsembleModel.load(str(model_path))
    else:
        canonical, _ = resolve_model_name(active_model)
        if canonical == 'lightgbm':
            from src.models.lightgbm_model import LightGBMModel
            model = LightGBMModel.load(str(model_path))
        elif canonical == 'randomforest':
            from src.models.randomforest_model import RandomForestMultiModel
            model = RandomForestMultiModel.load(str(model_path))
        elif canonical == 'mlp':
            from src.models.mlp_model import MLPModel
            model = MLPModel.load(str(model_path))

    target_type = cfg['training'].get('target_type', 'log_close')
    horizons    = cfg['training']['horizons']
    CHUNK_SIZE  = len(horizons)
    NUM_CHUNKS  = int(np.ceil(len(future_dates) / CHUNK_SIZE))
    feature_cols = model.feature_list
    seq_len      = None

    print(f"✅ Tabular 모델 로드: {active_model}")

print(f"   target_type: {target_type}  CHUNK_SIZE: {CHUNK_SIZE}  NUM_CHUNKS: {NUM_CHUNKS}")

## 5️⃣ Feature 생성 함수 정의

In [ ]:
def calculate_features_for_ticker(df_ticker: pd.DataFrame, config: dict) -> pd.DataFrame:
    """단일 종목 기술적 지표 재계산 (builder.py v3.6.0 규격)."""
    df = df_ticker.copy()
    params = config['preprocessing']

    for window in params['technical_windows']:
        ma = calc_sma(df['close'], window)
        df[f'feature_ma_{window}_disparity'] = (df['close'] / ma) - 1.0

    df['feature_volatility_20'] = df['close'].pct_change().rolling(20).std()
    df['feature_volume_ratio']  = calc_volume_ratio(df['volume'], params['volume_window'])
    df['feature_rsi_14']        = calc_rsi(df['close'], params['rsi_period'])

    macd, signal, hist = calc_macd(df['close'])
    df['feature_macd']        = macd
    df['feature_macd_signal'] = signal
    df['feature_macd_hist']   = hist

    upper, mid, lower = calc_bollinger(df['close'])
    df['feature_bb_pct_b'] = (df['close'] - lower) / (upper - lower + 1e-9)
    df['feature_bb_width']  = (upper - lower) / (mid + 1e-9)

    liquidity = (df['close'] * df['volume']).rolling(20).mean()
    df['feature_log_liquidity']  = np.log1p(liquidity)
    df['liquidity_score']        = liquidity
    _risk = df['feature_volatility_20'].fillna(0)
    df['feature_risk_composite'] = _risk
    df['risk_composite']         = _risk

    return df


def get_macro_row(pred_date: pd.Timestamp) -> dict:
    if pred_date in df_macro_lookup.index:
        row = df_macro_lookup.loc[pred_date]
    else:
        past = df_macro_lookup[df_macro_lookup.index <= pred_date]
        if past.empty:
            return {v: np.nan for v in MACRO_FEATURE_MAP.values()}
        row = past.iloc[-1]
    return {MACRO_FEATURE_MAP[col]: row[col] for col in macro_available_cols}


print("✅ Feature 함수 정의 완료")

## 6️⃣ Recursive Extension 예측

### 트랙별 핵심 차이

**Tabular 트랙**: 매 Chunk마다 `model.predict(X_1row)` — 단일 행(t) 입력  
**Seq 트랙**: 매 Chunk마다 `model.predict(X_seq_len_rows)` — seq_len 행 시퀀스 입력

역산 로직(`trapezoid_log_close`)과 피처 재계산은 동일합니다.

In [ ]:
pred_dates      = future_dates
all_forecasts   = []
skipped_tickers = []

for ticker in tqdm(df_features['ticker'].unique(), desc="종목별 예측"):
    df_ticker = df_features[df_features['ticker'] == ticker].copy()
    df_ticker = df_ticker.sort_values('date').reset_index(drop=True)

    is_kospi_val = df_ticker['feature_is_kospi'].iloc[-1] \
        if 'feature_is_kospi' in df_ticker.columns else 0

    try:
        if 'target_log_return_1d' in df_ticker.columns:
            prev_delta_y = float(df_ticker['target_log_return_1d'].iloc[-1])
        elif 'change_rate' in df_ticker.columns:
            prev_delta_y = float(np.log1p(df_ticker['change_rate'].iloc[-1]))
        elif 'change_pct' in df_ticker.columns:
            prev_delta_y = float(np.log1p(df_ticker['change_pct'].iloc[-1]))
        else:
            prev_delta_y = 0.0

        forecast_rows = []
        chunk_idx = 0

        while True:
            # ── 예측 입력 구성 ─────────────────────────────────────
            if USE_SEQ_TRACK:
                # Seq 트랙: seq_len 행 시퀀스
                if len(df_ticker) < seq_len:
                    break
                    
                # ✨ 수정: 추출한 윈도우(DataFrame)의 결측치를 보간합니다.
                # 1. ffill(): 이전 날짜의 정상 값으로 채움 (휴장일 대응)
                # 2. bfill(): 혹시 모를 맨 앞단 결측치 채움
                # 3. fillna(0): 그래도 남은 것은 0으로 안전하게 치환
                X_window_df = df_ticker[feature_cols].iloc[-seq_len:]
                X_window_df = X_window_df.ffill().bfill().fillna(0)
                
                X_window = X_window_df.values.astype(np.float32)
                
                if np.isnan(X_window).any(): # 보간을 했으므로 사실상 여기 걸리지 않습니다.
                    break
                    
                X_input = X_window[np.newaxis, :, :]   # (1, seq_len, n_features)
                raw_preds = model.predict(X_input).flatten()  # (forecast_horizon,)
            else:
                # Tabular 트랙: 단일 행
                X_input   = df_ticker[feature_cols].iloc[[-1]]
                raw_preds = model.predict(X_input)
                if isinstance(raw_preds, pd.DataFrame):
                    raw_preds = raw_preds.values
                raw_preds = raw_preds.flatten()

            # ── 역산 분기 ──────────────────────────────────────────
            last_row       = df_ticker.iloc[-1]
            close_base     = last_row['close']
            log_close_base = np.log(max(close_base, 1e-9))
            delta_y_t      = prev_delta_y

            n_steps = len(raw_preds)  # Tabular: len(horizons), Seq: forecast_horizon

            for h_idx in range(n_steps):
                global_idx = chunk_idx * n_steps + h_idx
                if global_idx >= len(pred_dates):
                    break
                pred_date = pred_dates[global_idx]

                if target_type == "log_return_1d":
                    pred_delta_h   = float(raw_preds[h_idx])
                    cum_log_return = float(np.sum(raw_preds[:h_idx + 1]))
                    pred_log_close = trapezoid_log_close(
                        log_close_base, cum_log_return, delta_y_t, pred_delta_h
                    )
                else:
                    pred_log_close = float(raw_preds[h_idx])

                pred_close = np.exp(pred_log_close)
                row = {
                    'date'           : pred_date,
                    'ticker'         : ticker,
                    'horizon'        : h_idx + 1,
                    'chunk_idx'      : chunk_idx,
                    'pred_log_close' : pred_log_close,
                    'pred_close'     : pred_close,
                }
                if SAVE_RAW_PREDICTIONS and target_type == "log_return_1d":
                    row['pred_log_return'] = float(raw_preds[h_idx])
                forecast_rows.append(row)

            # ── 다음 Chunk 기준가 및 앵커 갱신 ───────────────────
            if target_type == "log_return_1d":
                pred_delta_last     = float(raw_preds[-1])
                cum_log_return_all  = float(np.sum(raw_preds))
                last_pred_log_close = trapezoid_log_close(
                    log_close_base, cum_log_return_all, delta_y_t, pred_delta_last
                )
                last_pred_close = np.exp(last_pred_log_close)
                prev_delta_y    = pred_delta_last
            else:
                last_pred_log_close = float(raw_preds[-1])
                last_pred_close     = np.exp(last_pred_log_close)

            # df_ticker에 예측 행 추가
            pred_date_h1 = pred_dates[chunk_idx * n_steps] \
                if chunk_idx * n_steps < len(pred_dates) else None
            if pred_date_h1 is None:
                break

            new_row = {
                'date'   : pred_date_h1,
                'ticker' : ticker,
                'close'  : last_pred_close,
                'volume' : df_ticker['volume'].iloc[-20:].mean(),
                **get_macro_row(pred_date_h1),
                'feature_is_kospi'  : is_kospi_val,
                'feature_is_monday' : int(pred_date_h1.weekday() == 0),
                'feature_is_friday' : int(pred_date_h1.weekday() == 4),
            }
            df_ticker = pd.concat(
                [df_ticker, pd.DataFrame([new_row])], ignore_index=True
            )
            df_ticker = calculate_features_for_ticker(df_ticker, cfg)

            chunk_idx += 1
            if chunk_idx * n_steps >= len(pred_dates):
                break

        all_forecasts.extend(forecast_rows)

    except Exception as e:
        skipped_tickers.append((ticker, type(e).__name__, str(e)))

print(f"\n✅ 완료  성공: {df_features['ticker'].nunique() - len(skipped_tickers):,}  제외: {len(skipped_tickers)}")

## 7️⃣ 예측 결과 저장

In [ ]:
df_forecasts = pd.DataFrame(all_forecasts)
df_forecasts = df_forecasts.sort_values(['ticker', 'date']).reset_index(drop=True)

forecast_path = paths.get_forecasts_parquet()
df_forecasts.to_parquet(forecast_path, index=False)

print(f"💾 저장 완료: {forecast_path}")
print(f"   총 {len(df_forecasts):,}건  종목: {df_forecasts['ticker'].nunique():,}")
print(f"   컬럼: {df_forecasts.columns.tolist()}")
display(df_forecasts.head(10))

## 🏁 완료

### ✅ 산출물
```
data/04_forecasts/{ref_date}/{model_name}/future_forecasts.parquet
  컬럼: date, ticker, horizon, chunk_idx, pred_log_close, pred_close
  (debug.save_raw_predictions: true 시 pred_log_return 추가)
```

### 🔁 다음 단계
**`05_universe_selection.ipynb`** 실행

**v4.0.0**: Tabular / Seq 두 트랙 모두 동일한 `future_forecasts.parquet` 스키마를 출력합니다.